# Generalizability Figure With SSCD — U64

This notebook reproduces the U64 generalizability curve using the SSCD copy-detection score:

$$
GL = 1 - P\left(\max_i M_{SSCD}(x, y_i) > \tau\right), \quad \tau = 0.6.
$$

For each generated slice `x`, the notebook finds the nearest real training slice `y_i` in SSCD embedding space. A generated sample is counted as copy-like if its maximum SSCD cosine similarity is above the threshold. The plotted value is one SSCD generalization score per U64 model/run.

Important interpretation:

- Higher `GL` means fewer generated samples look like copies of the training set.
- Lower `GL` means more generated samples have close SSCD matches in the training set.
- This only measures copy-like behavior. Use P(k), histograms, and sample images separately for physics fidelity.


## Configuration

The default `RUN_MODE = "production"` uses all generated samples and all real training slices, which is the correct setting for the final figure. If the notebook is too slow or the kernel runs out of memory, change `RUN_MODE` to `"smoke"` first. Smoke mode is only for debugging and should not be used for the final number.

SSCD runs on CPU by default because the Great Lakes CUDA environment can raise NVRTC errors such as `failed to open libnvrtc-builtins.so.13.0`. To force GPU after fixing the CUDA environment, set `SSCD_DEVICE = "cuda"`.


In [ ]:
from __future__ import annotations

import json
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
import torch

# Robust project-root detection for Great Lakes and local copies of the notebook.
PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/home/jiamingp/Diffusion_model'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])

for candidate in (PROJECT_DIR, PROJECT_DIR / 'cosmo_diffusion'):
    if candidate.exists() and str(candidate) not in sys.path:
        sys.path.insert(0, str(candidate))

from simdiff_eval.io import as_nchw, load_npy, load_real_from_config
from simdiff_eval.sscd import load_sscd_torchscript, sscd_embeddings, sscd_generalization_metrics

ARCH = 'u64'
SEED = 123
THRESHOLD = 0.6
RENDER_MODE = 'fixed'
SSCD_IMAGE_SIZE = 320
SSCD_BATCH_SIZE = int(os.environ.get('SSCD_BATCH_SIZE', 16))
SIMILARITY_BATCH_SIZE = 256
SSCD_DEVICE = os.environ.get('SSCD_DEVICE', 'cpu')

# Use "production" for the final figure. Use "smoke" only to debug paths quickly.
RUN_MODE = os.environ.get('GENERALIZABILITY_MODE', 'production')  # "production" or "smoke"
if RUN_MODE == 'smoke':
    MAX_GENERATED = 64
    MAX_REAL_RAW_SIMS = 32
    MAX_REAL_SLICES = 1024
else:
    MAX_GENERATED = None
    MAX_REAL_RAW_SIMS = None
    MAX_REAL_SLICES = None

MANIFEST_CANDIDATES = [
    PROJECT_DIR / 'local' / 'fig1_lh' / 'manifest.json',
    PROJECT_DIR / 'configs' / 'templates' / 'reproducibility_manifest_template.json',
]
MANIFEST_PATH = next((p for p in MANIFEST_CANDIDATES if p.exists()), MANIFEST_CANDIDATES[0])
CONFIG_DIR = PROJECT_DIR / 'local' / 'fig1_lh' / 'configs'
SAMPLE_ROOTS = [
    PROJECT_DIR / 'results' / 'fig1_lh' / 'samples',
    PROJECT_DIR / 'results' / 'tables' / 'samples',
    PROJECT_DIR / 'results' / 'samples',
]
SSCD_PATH = Path(os.environ.get('SSCD_PATH', Path.home() / '.cache' / 'torch' / 'hub' / 'sscd_disc_mixup.torchscript.pt'))
OUTPUT_DIR = PROJECT_DIR / 'results' / 'figures'
TABLE_DIR = PROJECT_DIR / 'results' / 'tables'
CACHE_DIR = PROJECT_DIR / 'results' / 'cache' / 'sscd_generalizability_u64'

print('project:', PROJECT_DIR)
print('manifest:', MANIFEST_PATH, 'exists=', MANIFEST_PATH.exists())
print('config dir:', CONFIG_DIR, 'exists=', CONFIG_DIR.exists())
print('sample roots:')
for root in SAMPLE_ROOTS:
    print(' ', root, 'exists=', root.exists())
print('sscd:', SSCD_PATH, 'exists=', SSCD_PATH.exists())
print('run mode:', RUN_MODE)
print('sscd device:', SSCD_DEVICE)


## Discover U64 Runs

This table checks which U64 configs and generated sample files are available. Missing sample files need to be generated before the corresponding point can appear in the curve.

In [ ]:
def dataset_size(row: dict[str, Any]) -> float:
    for key in ('dataset_size', 'actual_2d', 'target_2d'):
        value = row.get(key)
        if value is not None:
            return float(value)
    raise ValueError(f"No dataset-size field in row: {row}")


def config_path_for(row: dict[str, Any]) -> Path:
    if row.get('config'):
        path = Path(row['config'])
        if not path.is_absolute():
            path = PROJECT_DIR / path
        return path
    return CONFIG_DIR / f"{row['run_name']}.yaml"


def sample_path_for(row: dict[str, Any]) -> Path | None:
    if row.get('sample_path'):
        raw = str(row['sample_path']).format(seed=SEED, run_name=row['run_name'])
        path = Path(raw)
        if not path.is_absolute():
            path = PROJECT_DIR / path
        if path.exists():
            return path
    for root in SAMPLE_ROOTS:
        path = root / f"{row['run_name']}_seed{SEED}.npy"
        if path.exists():
            return path
    return None

manifest = json.loads(MANIFEST_PATH.read_text())
rows = sorted([row for row in manifest if row.get('arch') == ARCH], key=dataset_size)
if not rows:
    raise RuntimeError(f'No manifest rows found for ARCH={ARCH!r}')

status_rows = []
for row in rows:
    config_path = config_path_for(row)
    sample_path = sample_path_for(row)
    status_rows.append({
        'run_name': row['run_name'],
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'config_exists': config_path.exists(),
        'sample_exists': sample_path is not None,
        'config_path': str(config_path),
        'sample_path': str(sample_path) if sample_path is not None else None,
    })

status_df = pd.DataFrame(status_rows)
display(status_df)


## Compute SSCD Generalization Scores

This cell computes one SSCD score per U64 run. It uses the exact training config to load and normalize the real training slices, then compares generated embeddings against real-training embeddings.

The output columns to watch are:

- `generalization_score`: final plotted value, higher means fewer near-copies.
- `copy_fraction`: fraction of generated samples with nearest-training SSCD similarity above `THRESHOLD`.
- `max_similarity_median`: median nearest-training SSCD similarity, useful as a continuous diagnostic.


In [ ]:
def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return arr[idx].copy()


def cache_label(run_name: str, kind: str) -> str:
    real_raw = 'all' if MAX_REAL_RAW_SIMS is None else str(MAX_REAL_RAW_SIMS)
    real_slices = 'all' if MAX_REAL_SLICES is None else str(MAX_REAL_SLICES)
    gen = 'all' if MAX_GENERATED is None else str(MAX_GENERATED)
    return (
        f'{run_name}_{kind}_seed{SEED}_thr{THRESHOLD}_'
        f'{RENDER_MODE}_img{SSCD_IMAGE_SIZE}_gen{gen}_raw{real_raw}_real{real_slices}.pt'
    )


def embed_with_cache(images: np.ndarray, model: torch.nn.Module, *, run_name: str, kind: str) -> torch.Tensor:
    CACHE_DIR.mkdir(parents=True, exist_ok=True)
    path = CACHE_DIR / cache_label(run_name, kind)
    if path.exists():
        return torch.load(path, map_location='cpu')
    emb = sscd_embeddings(
        images,
        model,
        device=SSCD_DEVICE,
        batch_size=SSCD_BATCH_SIZE,
        image_size=SSCD_IMAGE_SIZE,
        render_mode=RENDER_MODE,
    )
    torch.save(emb, path)
    return emb

if not SSCD_PATH.exists():
    raise FileNotFoundError(
        f'SSCD checkpoint not found: {SSCD_PATH}\n'
        'Download it with:\n'
        'curl -L -o ~/.cache/torch/hub/sscd_disc_mixup.torchscript.pt '
        'https://dl.fbaipublicfiles.com/sscd-copy-detection/sscd_disc_mixup.torchscript.pt'
    )

sscd_model = load_sscd_torchscript(SSCD_PATH, device=SSCD_DEVICE)
print('loaded SSCD on', SSCD_DEVICE)

records = []
for row in rows:
    run_name = row['run_name']
    sample_path = sample_path_for(row)
    config_path = config_path_for(row)
    if sample_path is None or not config_path.exists():
        print('SKIP missing inputs:', run_name, 'sample=', sample_path, 'config exists=', config_path.exists())
        continue

    generated = as_nchw(load_npy(sample_path))
    generated = evenly_limit(generated, MAX_GENERATED)

    real_training = load_real_from_config(config_path, max_raw_samples=MAX_REAL_RAW_SIMS)
    real_training = evenly_limit(as_nchw(real_training), MAX_REAL_SLICES)

    print(f'{run_name}: generated={len(generated)} real_training={len(real_training)} dataset_size={dataset_size(row):.0f}')
    gen_emb = embed_with_cache(generated, sscd_model, run_name=run_name, kind='generated')
    real_emb = embed_with_cache(real_training, sscd_model, run_name=run_name, kind='real')
    metrics = sscd_generalization_metrics(
        gen_emb,
        real_emb,
        threshold=THRESHOLD,
        batch_size=SIMILARITY_BATCH_SIZE,
    )

    record = {
        'run_name': run_name,
        'arch': ARCH,
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'n_generated': len(generated),
        'n_real_training_compared': len(real_training),
        'run_mode': RUN_MODE,
        'max_generated': MAX_GENERATED,
        'max_real_raw_sims': MAX_REAL_RAW_SIMS,
        'max_real_slices': MAX_REAL_SLICES,
        'render_mode': RENDER_MODE,
        **metrics,
    }
    records.append(record)
    print(
        f"  GL={record['generalization_score']:.3f} "
        f"copy_fraction={record['copy_fraction']:.3f} "
        f"max_sim_median={record['max_similarity_median']:.3f}"
    )

if not records:
    raise RuntimeError('No SSCD records were computed. Check the status table for missing sample/config files.')

df = pd.DataFrame(records).sort_values('dataset_size').reset_index(drop=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
out_csv = TABLE_DIR / f'generalizability_sscd_{ARCH}.csv'
df.to_csv(out_csv, index=False)
print('saved', out_csv)
display(df)


## Plot The U64 SSCD Generalizability Curve

This is the main plot: one SSCD generalization score per U64 model as a function of training dataset size.

In [ ]:
def xfmt(x: float, _pos: int) -> str:
    if x <= 0:
        return ''
    exponent = int(round(np.log2(x)))
    if np.isclose(x, 2**exponent):
        return rf'$2^{{{exponent}}}$'
    return f'{x:g}'

plot_df = df.sort_values('dataset_size')
fig, ax = plt.subplots(figsize=(7.2, 5.0))
ax.plot(
    plot_df['dataset_size'],
    plot_df['generalization_score'],
    color='steelblue',
    marker='o',
    lw=2.5,
    ms=7,
)
ax.set_xscale('log', base=2)
ax.set_ylim(-0.05, 1.05)
ax.set_xlabel('dataset size (2D training slices)')
ax.set_ylabel(r'SSCD generalization score, $1 - f_{copy}$')
ax.set_title(f'UNet-64 SSCD generalizability, threshold={THRESHOLD}')
ax.xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
ax.grid(alpha=0.25)

if RUN_MODE != 'production' or MAX_REAL_RAW_SIMS is not None or MAX_REAL_SLICES is not None or MAX_GENERATED is not None:
    ax.text(
        0.02,
        0.03,
        'approximate smoke-test caps are active',
        transform=ax.transAxes,
        fontsize=9,
        color='crimson',
    )

fig.tight_layout()
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
out_png = OUTPUT_DIR / f'generalizability_sscd_{ARCH}.png'
out_pdf = OUTPUT_DIR / f'generalizability_sscd_{ARCH}.pdf'
fig.savefig(out_png, dpi=180, bbox_inches='tight')
fig.savefig(out_pdf, bbox_inches='tight')
print('saved', out_png)
print('saved', out_pdf)
plt.show()


## Diagnostic: Copy Fraction And Max Similarity

The main score is `generalization_score = 1 - copy_fraction`. The diagnostic plot below is useful for checking why the score changes.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharex=True)
plot_df = df.sort_values('dataset_size')

axes[0].plot(plot_df['dataset_size'], plot_df['copy_fraction'], marker='o', lw=2, color='tab:red')
axes[0].set_xscale('log', base=2)
axes[0].set_ylim(-0.05, 1.05)
axes[0].set_xlabel('dataset size (2D training slices)')
axes[0].set_ylabel('copy fraction')
axes[0].set_title(f'fraction above SSCD threshold {THRESHOLD}')
axes[0].xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
axes[0].grid(alpha=0.25)

axes[1].plot(plot_df['dataset_size'], plot_df['max_similarity_median'], marker='o', lw=2, label='median')
axes[1].plot(plot_df['dataset_size'], plot_df['max_similarity_q90'], marker='s', lw=2, label='q90')
axes[1].axhline(THRESHOLD, color='black', linestyle=':', lw=1.5, label='threshold')
axes[1].set_xscale('log', base=2)
axes[1].set_xlabel('dataset size (2D training slices)')
axes[1].set_ylabel('nearest-training SSCD similarity')
axes[1].set_title('nearest-neighbor similarity diagnostics')
axes[1].xaxis.set_major_formatter(ticker.FuncFormatter(xfmt))
axes[1].grid(alpha=0.25)
axes[1].legend()

fig.tight_layout()
out_diag = OUTPUT_DIR / f'generalizability_sscd_{ARCH}_diagnostics.png'
fig.savefig(out_diag, dpi=180, bbox_inches='tight')
print('saved', out_diag)
plt.show()


## Command-Line Equivalent

The notebook and script use the same SSCD helpers. For a full non-interactive run from the repo root on Great Lakes:

```bash
export SSCD_DEVICE=cpu
python scripts/plot_generalizability_sscd.py \
  --arch u64 \
  --manifest local/fig1_lh/manifest.json \
  --config-dir local/fig1_lh/configs \
  --sample-root results/fig1_lh/samples \
  --sscd-path ~/.cache/torch/hub/sscd_disc_mixup.torchscript.pt \
  --threshold 0.6 \
  --output-csv results/tables/generalizability_sscd_u64.csv \
  --output-figure results/figures/generalizability_sscd_u64.png
```

If samples were saved under `results/tables/samples`, change `--sample-root` accordingly.
